# ❤️ Heart Disease Prediction Using Patient Health Data

---

**End-to-End Machine Learning Pipeline**

| Detail | Value |
|--------|-------|
| **Dataset** | Cardiovascular Disease Dataset (Kaggle) |
| **Records** | 70,000 patients |
| **Features** | 11 clinical parameters + 1 target |
| **Models** | Logistic Regression, Random Forest, Decision Tree, SVM, XGBoost |
| **Target** | Cardiovascular Disease (binary: 0/1) |
| **Author** | Capstone Project — Predictive Analytics |

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)

# XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed. Skipping XGBoost model.")

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(['#e53935', '#ff6b6b', '#4facfe', '#43e97b', '#f093fb', '#f7971e'])

print("All libraries imported successfully!")

---
## 2. Load Dataset

The **Cardiovascular Disease Dataset** from Kaggle contains **70,000 patient records** with 11 clinical features and 1 binary target variable.

**Source:** https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset

In [ ]:
df = pd.read_csv('../data/heart.csv')
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

---
## 3. Data Overview

### 3.1 Dataset Shape & Data Types

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nData Types:")
df.dtypes

### 3.2 Descriptive Statistics

In [ ]:
df.describe().round(2)

### 3.3 Missing Values & Duplicates

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nDuplicated rows: {df.duplicated().sum()}")

---
## 4. Data Cleaning

Steps:
1. Drop the `id` column (not a feature)
2. Convert `age` from days to years
3. Remove physiologically impossible blood pressure outliers
4. Filter extreme height/weight outliers (1st–99th percentile)
5. Remove duplicate records

In [ ]:
# Drop ID column
if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)
    print("Dropped 'id' column")

# Convert age from days to years
if df['age'].max() > 200:
    df['age'] = (df['age'] / 365.25).astype(int)
    print(f"Converted age to years. Range: {df['age'].min()} - {df['age'].max()}")

print(f"\nShape before cleaning: {df.shape}")

In [ ]:
# Remove blood pressure outliers
# Systolic: 60-250 mmHg, Diastolic: 40-200 mmHg
df = df[(df['ap_hi'] >= 60) & (df['ap_hi'] <= 250)]
df = df[(df['ap_lo'] >= 40) & (df['ap_lo'] <= 200)]

# Systolic must be greater than diastolic
df = df[df['ap_hi'] > df['ap_lo']]

# Remove extreme height/weight outliers (1st-99th percentile)
for col in ['height', 'weight']:
    q_low = df[col].quantile(0.01)
    q_hi = df[col].quantile(0.99)
    df = df[(df[col] >= q_low) & (df[col] <= q_hi)]
    print(f"{col}: kept range [{q_low:.0f}, {q_hi:.0f}]")

# Remove duplicates
n_dupes = df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nRemoved {n_dupes} duplicates")
print(f"Shape after cleaning: {df.shape}")

### 4.1 Target Distribution

In [ ]:
print("Target Distribution:")
print(df['cardio'].value_counts())
print(f"\nDisease Rate: {df['cardio'].mean():.2%}")

---
## 5. Exploratory Data Analysis (EDA)

### 5.1 Create Readable Labels

In [ ]:
df_eda = df.copy()
df_eda['gender_label'] = df_eda['gender'].map({1: 'Female', 2: 'Male'})
df_eda['chol_label'] = df_eda['cholesterol'].map({1: 'Normal', 2: 'Above Normal', 3: 'Well Above Normal'})
df_eda['gluc_label'] = df_eda['gluc'].map({1: 'Normal', 2: 'Above Normal', 3: 'Well Above Normal'})
df_eda['smoke_label'] = df_eda['smoke'].map({1: 'Yes', 0: 'No'})
df_eda['alco_label'] = df_eda['alco'].map({1: 'Yes', 0: 'No'})
df_eda['active_label'] = df_eda['active'].map({1: 'Yes', 0: 'No'})
df_eda['cardio_label'] = df_eda['cardio'].map({1: 'Disease', 0: 'No Disease'})
df_eda['bmi'] = df_eda['weight'] / ((df_eda['height'] / 100) ** 2)

# Define color scheme
colors = ['#43e97b', '#ff416c']
palette = {'No Disease': '#43e97b', 'Disease': '#ff416c'}

print("Labels created.")
df_eda.head()

### 5.2 Target Distribution (Donut Chart)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    df['cardio'].value_counts().values,
    labels=['No Disease', 'Disease'],
    colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.85,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
    textprops={'fontsize': 13, 'fontweight': 'bold'}
)
ax.set_title('Cardiovascular Disease Distribution (70K Patients)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../models/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Age Distribution by Disease

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for label, color in zip(['No Disease', 'Disease'], colors):
    sub = df_eda[df_eda['cardio_label'] == label]
    axes[0].hist(sub['age'], bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
axes[0].set_xlabel('Age (years)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Age Distribution by Disease', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Box plot
sns.boxplot(data=df_eda, x='cardio_label', y='age', palette=palette, ax=axes[1])
axes[1].set_xlabel('Diagnosis', fontsize=12)
axes[1].set_ylabel('Age (years)', fontsize=12)
axes[1].set_title('Age by Diagnosis', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../models/eda_age.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Blood Pressure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_eda, x='cardio_label', y='ap_hi', palette=palette, ax=axes[0])
axes[0].set_xlabel('Diagnosis', fontsize=12)
axes[0].set_ylabel('Systolic BP (mmHg)', fontsize=12)
axes[0].set_title('Systolic Blood Pressure by Disease', fontsize=14, fontweight='bold')

sns.boxplot(data=df_eda, x='cardio_label', y='ap_lo', palette=palette, ax=axes[1])
axes[1].set_xlabel('Diagnosis', fontsize=12)
axes[1].set_ylabel('Diastolic BP (mmHg)', fontsize=12)
axes[1].set_title('Diastolic Blood Pressure by Disease', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../models/eda_blood_pressure.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Disease by Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender count
gender_d = df_eda.groupby(['gender_label', 'cardio_label']).size().reset_index(name='Count')
sns.barplot(data=gender_d, x='gender_label', y='Count', hue='cardio_label', palette=palette, ax=axes[0])
axes[0].set_title('Heart Disease by Gender', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')

# Disease rate by gender
gender_rate = df_eda.groupby('gender_label')['cardio'].mean() * 100
gender_rate.plot(kind='bar', color=['#f093fb', '#4facfe'], edgecolor='white', ax=axes[1])
axes[1].set_title('Disease Rate by Gender (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Disease Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(gender_rate):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../models/eda_gender.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Cholesterol & BMI Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cholesterol
chol_d = df_eda.groupby(['chol_label', 'cardio_label']).size().reset_index(name='Count')
sns.barplot(data=chol_d, x='chol_label', y='Count', hue='cardio_label', palette=palette, ax=axes[0])
axes[0].set_title('Disease by Cholesterol Level', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cholesterol Level')

# BMI
sns.boxplot(data=df_eda, x='cardio_label', y='bmi', palette=palette, ax=axes[1])
axes[1].set_title('BMI by Diagnosis', fontsize=14, fontweight='bold')
axes[1].set_ylabel('BMI (kg/m\u00b2)')
axes[1].set_xlabel('Diagnosis')

plt.tight_layout()
plt.savefig('../models/eda_chol_bmi.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Lifestyle Factors (Smoking, Alcohol, Physical Activity)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lifestyle_features = [
    ('smoke_label', 'Smoking'),
    ('alco_label', 'Alcohol Intake'),
    ('active_label', 'Physical Activity'),
]

for i, (col, title) in enumerate(lifestyle_features):
    rate = df_eda.groupby(col)['cardio'].mean() * 100
    rate.plot(kind='bar', color=['#e53935', '#ff6b6b'], edgecolor='white', ax=axes[i])
    axes[i].set_title(f'Disease Rate by {title}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Disease Rate (%)')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)
    for j, v in enumerate(rate):
        axes[i].text(j, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../models/eda_lifestyle.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Disease Rate Across All Categorical Features

In [ ]:
cat_features = {
    'Gender': 'gender_label',
    'Cholesterol': 'chol_label',
    'Glucose': 'gluc_label',
    'Smoking': 'smoke_label',
    'Alcohol': 'alco_label',
    'Physical Activity': 'active_label',
}

rates = []
for feat_name, col_name in cat_features.items():
    r = df_eda.groupby(col_name)['cardio'].mean() * 100
    for cat_val, pct in r.items():
        rates.append({'Feature': feat_name, 'Category': str(cat_val), 'Disease Rate (%)': round(pct, 1)})

rate_df = pd.DataFrame(rates)
rate_pivot = rate_df.pivot(index='Feature', columns='Category', values='Disease Rate (%)')

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
sns.heatmap(rate_pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', center=50,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Disease Rate (%)'})
ax.set_title('Disease Rate (%) Across All Categories', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/eda_disease_rate_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.9 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../models/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.10 Correlation with Target (Cardiovascular Disease)

In [ ]:
target_corr = corr['cardio'].drop('cardio').sort_values(ascending=True)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
bars = ax.barh(target_corr.index, target_corr.values,
               color=['#ff416c' if v > 0 else '#43e97b' for v in target_corr.values],
               edgecolor='white', linewidth=0.5)
ax.set_xlabel('Correlation Coefficient', fontsize=12)
ax.set_title('Correlation with Cardiovascular Disease (Target)', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/eda_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Feature Engineering

Creating derived medical features:

| Feature | Formula | Medical Significance |
|---------|---------|---------------------|
| `bmi` | weight / (height/100)² | Body Mass Index — obesity indicator |
| `pulse_pressure` | systolic − diastolic | Arterial stiffness |
| `map_bp` | diastolic + ⅓(systolic − diastolic) | Mean Arterial Pressure |
| `bp_ratio` | systolic / diastolic | Blood pressure ratio |
| `age_bmi` | age × bmi | Age-obesity interaction |

In [ ]:
# BMI = weight(kg) / height(m)^2
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# Pulse Pressure = Systolic - Diastolic
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

# Mean Arterial Pressure = Diastolic + 1/3 * (Systolic - Diastolic)
df['map_bp'] = df['ap_lo'] + (df['ap_hi'] - df['ap_lo']) / 3

# BP ratio
df['bp_ratio'] = df['ap_hi'] / df['ap_lo']

# Age * BMI interaction
df['age_bmi'] = df['age'] * df['bmi']

print(f"Shape after feature engineering: {df.shape}")
print(f"New features: bmi, pulse_pressure, map_bp, bp_ratio, age_bmi")
df[['bmi', 'pulse_pressure', 'map_bp', 'bp_ratio', 'age_bmi']].describe().round(2)

---
## 7. Data Preprocessing

### 7.1 Separate Features and Target

In [ ]:
X = df.drop('cardio', axis=1)
y = df['cardio']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

### 7.2 One-Hot Encoding

In [ ]:
# One-hot encode cholesterol and glucose (ordinal categories: 1, 2, 3)
cat_cols = ['cholesterol', 'gluc']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=float)

print(f"Shape after encoding: {X.shape}")
print(f"Columns: {X.columns.tolist()}")

### 7.3 Feature Scaling (StandardScaler)

In [ ]:
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo',
            'bmi', 'pulse_pressure', 'map_bp', 'bp_ratio', 'age_bmi']

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
X = X.astype(float)
feature_names = X.columns.tolist()

print(f"Scaled {len(num_cols)} numerical features")
print(f"Total features: {len(feature_names)}")
X.head()

### 7.4 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")
print(f"Disease rate (train): {y_train.mean():.2%}")
print(f"Disease rate (test):  {y_test.mean():.2%}")

---
## 8. Model Training

Training **5 classification models** on the cardiovascular disease dataset:

| # | Model | Type |
|---|-------|------|
| 1 | Logistic Regression | Linear |
| 2 | Random Forest | Ensemble (Bagging) |
| 3 | Decision Tree | Tree-based |
| 4 | SVM | Kernel-based |
| 5 | XGBoost | Ensemble (Boosting) |

> **Note:** SVM is subsampled to 10,000 training points (too slow on 50K+ samples)

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, solver='lbfgs', n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_split=10, min_samples_leaf=5, random_state=42
    ),
    'SVM': SVC(
        kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42
    ),
}

if HAS_XGBOOST:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss', n_jobs=-1
    )

print(f"Models to train: {list(models.keys())}")

In [ ]:
results = {}
all_metrics = []

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  >> Training: {name}")
    print(f"{'='*55}")

    # SVM subsample (too slow on 50K+ samples)
    if name == 'SVM' and X_train.shape[0] > 10000:
        print(f"  Subsampling to 10,000 for SVM (original: {X_train.shape[0]:,})")
        rng = np.random.RandomState(42)
        idx = rng.choice(X_train.shape[0], 10000, replace=False)
        X_tr, y_tr = X_train[idx], y_train[idx]
    else:
        X_tr, y_tr = X_train, y_train

    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - start

    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Train Time (s)': round(train_time, 2),
    }
    all_metrics.append(metrics)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'metrics': metrics}

    print(f"  Time: {train_time:.2f}s")
    print(f"  Accuracy:  {metrics['Accuracy']:.4f}")
    print(f"  Precision: {metrics['Precision']:.4f}")
    print(f"  Recall:    {metrics['Recall']:.4f}")
    print(f"  F1-Score:  {metrics['F1-Score']:.4f}")
    print(f"  ROC-AUC:   {metrics['ROC-AUC']:.4f}")

### 8.1 Classification Reports

In [ ]:
for name, data in results.items():
    print(f"\n{'='*50}")
    print(f"  Classification Report: {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, data['y_pred'], target_names=['No Disease', 'Disease']))

### 8.2 Metrics Comparison Table

In [ ]:
metrics_df = pd.DataFrame(all_metrics)

# Highlight best model
best_idx = metrics_df['F1-Score'].idxmax()
best_name = metrics_df.loc[best_idx, 'Model']

print(f"\n{'#'*50}")
print(f"  BEST MODEL: {best_name}")
print(f"  F1-Score: {metrics_df.loc[best_idx, 'F1-Score']:.4f}")
print(f"  ROC-AUC:  {metrics_df.loc[best_idx, 'ROC-AUC']:.4f}")
print(f"{'#'*50}")

metrics_df.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
                                color='#43e97b')

---
## 9. Model Evaluation

### 9.1 Performance Metrics Comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_df))
width = 0.15
bar_colors = ['#e53935', '#ff6b6b', '#f093fb', '#4facfe', '#43e97b']

for i, metric in enumerate(metric_cols):
    bars = ax.bar(x + i * width, metrics_df[metric], width,
                  label=metric, color=bar_colors[i], edgecolor='white', linewidth=0.5)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_df['Model'], fontsize=11, rotation=15, ha='right')
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])
plt.tight_layout()
plt.savefig('../models/eval_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 ROC Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 7))
model_colors = ['#e53935', '#ff6b6b', '#f093fb', '#4facfe', '#43e97b']

for i, (name, data) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, data['y_prob'])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=model_colors[i], lw=2.5,
            label=f'{name} (AUC = {roc_auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves \u2014 All Models', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../models/eval_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Confusion Matrices

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

for i, (name, data) in enumerate(results.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                ax=axes[i], cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[i].set_title(name, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../models/eval_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Feature Importance (Best Model)

In [ ]:
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    sorted_idx = np.argsort(importance)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.barh(range(len(sorted_idx)), importance[sorted_idx],
            color=['#e53935' if imp > np.mean(importance) else '#ffcdd2'
                   for imp in importance[sorted_idx]],
            edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=10)
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(f'Feature Importance \u2014 {best_name}', fontsize=16, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/eval_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Top 5 features
    print(f"\nTop 5 Most Important Features ({best_name}):")
    top5 = sorted_idx[-5:][::-1]
    for rank, idx in enumerate(top5, 1):
        print(f"  {rank}. {feature_names[idx]}: {importance[idx]:.4f}")

elif hasattr(best_model, 'coef_'):
    coefs = best_model.coef_[0]
    sorted_idx = np.argsort(np.abs(coefs))
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    colors_bar = ['#43e97b' if c > 0 else '#ff416c' for c in coefs[sorted_idx]]
    ax.barh(range(len(sorted_idx)), coefs[sorted_idx], color=colors_bar,
            edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=10)
    ax.set_xlabel('Coefficient Value', fontsize=12)
    ax.set_title(f'Feature Coefficients \u2014 {best_name}', fontsize=16, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/eval_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"Feature importance not available for {best_name}")

---
## 10. Cross Validation (5-Fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

print("5-Fold Cross Validation Results:")
print(f"{'Model':<25s} | {'Mean F1':>8s} | {'Std F1':>8s}")
print("-" * 50)

for name, data in results.items():
    model_obj = data['model']
    
    # Use subsample for SVM CV too
    if name == 'SVM' and X.shape[0] > 15000:
        rng = np.random.RandomState(42)
        idx = rng.choice(X.shape[0], 15000, replace=False)
        X_cv, y_cv = X.values[idx], y.values[idx]
    else:
        X_cv, y_cv = X.values, y.values
    
    scores = cross_val_score(model_obj, X_cv, y_cv, cv=cv, scoring='f1')
    cv_results.append({'Model': name, 'CV Mean F1': scores.mean(), 'CV Std F1': scores.std()})
    print(f"  {name:<23s} | {scores.mean():>8.4f} | {scores.std():>8.4f}")

cv_df = pd.DataFrame(cv_results)
cv_df

---
## 11. Save Models

Saving the best model, scaler, and metadata for the Streamlit dashboard.

In [ ]:
save_dir = '../models'
os.makedirs(save_dir, exist_ok=True)

# Save best model
joblib.dump(results[best_name]['model'], os.path.join(save_dir, 'best_model.pkl'))
print(f"Saved best_model.pkl ({best_name})")

# Save scaler
joblib.dump(scaler, os.path.join(save_dir, 'preprocessor.pkl'))
print("Saved preprocessor.pkl")

# Save metadata
metadata = {
    'best_model_name': best_name,
    'feature_names': feature_names,
    'metrics': metrics_df.to_dict('records'),
}
joblib.dump(metadata, os.path.join(save_dir, 'model_metadata.pkl'))
print("Saved model_metadata.pkl")

# Save all results
results_to_save = {
    name: {'model': data['model'], 'y_pred': data['y_pred'], 'y_prob': data['y_prob']}
    for name, data in results.items()
}
joblib.dump(results_to_save, os.path.join(save_dir, 'all_results.pkl'))
print("Saved all_results.pkl")

print(f"\nAll models saved to {save_dir}/")

---
## 12. Summary

### Key Results

In [ ]:
print(f"""
{'='*60}
  PROJECT SUMMARY
{'='*60}

Dataset:     Cardiovascular Disease (Kaggle)
Records:     {len(df):,} patients (after cleaning)
Features:    {len(feature_names)} (after engineering + encoding)
Models:      {', '.join(results.keys())}
Best Model:  {best_name}
Best F1:     {metrics_df.loc[best_idx, 'F1-Score']:.4f}
Best AUC:    {metrics_df.loc[best_idx, 'ROC-AUC']:.4f}

Key Findings:
  - Systolic blood pressure is the strongest predictor
  - Higher BMI correlates with elevated disease risk
  - Cholesterol 'Well Above Normal' nearly doubles disease rates
  - Age is a significant risk factor
  - Physical activity provides protective effects

Saved Artifacts:
  - models/best_model.pkl ({best_name})
  - models/preprocessor.pkl (StandardScaler)
  - models/model_metadata.pkl
  - models/all_results.pkl

Next Step:
  streamlit run app.py
""")

In [ ]:
# Final metrics table
metrics_df.style.highlight_max(
    subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    color='#43e97b'
).format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'ROC-AUC': '{:.4f}',
    'Train Time (s)': '{:.2f}',
})